# Week 8 Lab 2: Midterm Work Session

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_08/materials/notebooks/midterm_work_session.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_08/materials/notebooks/midterm_work_session.ipynb)

**This notebook is classwork. It is not the midterm.** The midterm is still two
files, `midterm.sql` and `README.md`, in one ZIP in Brightspace by **Tuesday,
November 3, 2026**. Read the [midterm page](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_08/materials/assignments/midterm_project.md) for the full requirements.

This notebook runs the midterm starter in a temporary PostgreSQL database inside
Colab, so you can see **every** result in order. You work through Parts 1, 2, 4,
and 5 here. As you go, copy your SQL into `midterm.sql` and your observed results
into `README.md`.

No Supabase password is needed. If you have not set up a Supabase project, this
notebook's PostgreSQL counts as a local practice database for the midterm. Name
it in your README as "PostgreSQL in a Colab runtime".

| Step | What you do | Minutes |
|---|---|---:|
| 0 | Start PostgreSQL and load the starter | 5 |
| 1 | Part 1: check the data and test three integrity rules | 20 |
| 2 | Part 2: roll back one change, commit another | 15 |
| 3 | Part 4: run the index experiment | 15 |
| 4 | Part 5: back up, restore somewhere separate, check | 20 |
| 5 | Record where you are, then clean up | 5 |

Part 3 (blocking) uses the Week 5 notebook and is not in this lab.

**Run the cells in order.** Later cells use names created by earlier cells.

## Step 0: Start PostgreSQL

This is the same startup cell as the Day 1 recovery notebook. In Colab it
installs PostgreSQL and starts it. It also finds client tools (`psql`,
`pg_dump`, `pg_restore`) that match the server's version. It takes about a
minute the first time.

In [ ]:
import hashlib
import os
from pathlib import Path
import re
import shutil
import subprocess

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if not IN_COLAB and os.environ.get("PGHOST", "") and not os.environ["PGHOST"].startswith("/"):
    raise RuntimeError("Use a local PostgreSQL Unix socket, not a remote host, for this practice notebook.")
if not IN_COLAB and (os.environ.get("PGSERVICE") or os.environ.get("PGHOSTADDR")):
    raise RuntimeError("Unset PGSERVICE and PGHOSTADDR so the local practice target is explicit.")

if IN_COLAB and shutil.which("pg_dump") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"],
        check=True,
    )

if IN_COLAB:
    subprocess.run(["service", "postgresql", "start"], check=True)

PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
print("Running in Colab:", IN_COLAB)
print("PostgreSQL command prefix:", PG_PREFIX or "current local user")

readiness = subprocess.run(
    PG_PREFIX + ["pg_isready"],
    check=True,
    text=True,
    capture_output=True,
)
print(readiness.stdout.strip())

server_version_result = subprocess.run(
    PG_PREFIX + ["psql", "-X", "--set=ON_ERROR_STOP=on", "--dbname", "postgres",
                 "--tuples-only", "--no-align", "--command", "SHOW server_version_num"],
    check=True,
    text=True,
    capture_output=True,
)
server_version_num = int(server_version_result.stdout.strip())
server_major = server_version_num // 10000

candidate_directories = [
    Path(f"/usr/lib/postgresql/{server_major}/bin"),
    Path(f"/opt/homebrew/opt/postgresql@{server_major}/bin"),
    Path(f"/Applications/Postgres.app/Contents/Versions/{server_major}/bin"),
]
path_pg_dump = shutil.which("pg_dump")
if path_pg_dump:
    candidate_directories.append(Path(path_pg_dump).parent)

PG_BIN = None
for candidate_directory in candidate_directories:
    candidate_dump = candidate_directory / "pg_dump"
    if not candidate_dump.exists():
        continue
    version_text = subprocess.run(
        [str(candidate_dump), "--version"],
        check=True,
        text=True,
        capture_output=True,
    ).stdout
    version_match = re.search(r"(\d+)(?:\.\d+)?", version_text)
    if version_match and int(version_match.group(1)) == server_major:
        PG_BIN = candidate_directory
        break

if PG_BIN is None:
    raise RuntimeError(
        f"For this same-server restore, install PostgreSQL client major version {server_major} "
        "to match the server, then rerun this cell."
    )

PSQL = str(PG_BIN / "psql")
CREATEDB = str(PG_BIN / "createdb")
DROPDB = str(PG_BIN / "dropdb")
PG_DUMP = str(PG_BIN / "pg_dump")
PG_RESTORE = str(PG_BIN / "pg_restore")

print("Server major version:", server_major)
print("Compatible client directory:", PG_BIN)
print(subprocess.run([PG_DUMP, "--version"], check=True, text=True, capture_output=True).stdout.strip())

## Step 0, continued: Load the Midterm Starter

This cell creates a practice database and runs **SECTION 1** of
`midterm_starter.sql`, the same file you saved as `midterm.sql`. SECTION 1 builds
the three tables, loads the data, and counts the rows.

It also defines `run_sql(...)`. Put SQL inside it and it prints every statement
and every result in order. Errors show their SQLSTATE code, such as `23505`.

**To reset everything,** rerun this cell. SECTION 1 drops and rebuilds the
`midterm_metro` schema, so every change you made is lost.

**You should see** counts of **users 8, tickets 12, ticket_events 21**.

In [ ]:
from urllib.request import urlopen
from uuid import uuid4
import tempfile

STARTER_URL = ("https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/"
               "main/week_08/materials/assignments/midterm_starter.sql")
local_starter = Path("../assignments/midterm_starter.sql")
if local_starter.exists():
    starter_text = local_starter.read_text(encoding="utf-8")
else:
    starter_text = urlopen(STARTER_URL, timeout=30).read().decode("utf-8")

section_1 = starter_text.split("-- TODO: Add a query joining", 1)[0]
assert "CREATE TABLE ticket_events" in section_1

if "PRACTICE_DB" not in globals():
    run_id = uuid4().hex[:8]
    PRACTICE_DB = "cst4714_midterm_practice_" + run_id
    RESTORE_DB = "cst4714_midterm_restore_" + run_id
    ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="cst4714-midterm-"))
    if IN_COLAB:
        shutil.chown(ARTIFACT_DIR, user="postgres", group="postgres")
    DUMP_FILE = ARTIFACT_DIR / "midterm_metro.dump"
    subprocess.run(PG_PREFIX + [CREATEDB, PRACTICE_DB], check=True)


def run_sql(sql_text, database=None):
    """Run SQL in one psql session and print every statement and result."""
    result = subprocess.run(
        PG_PREFIX + [PSQL, "-X", "--echo-queries", "--set=ON_ERROR_STOP=on",
                     "--set=VERBOSITY=verbose", "--dbname", database or PRACTICE_DB,
                     "--file", "-"],
        input=sql_text, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print(result.stdout.strip("
"))
    if result.returncode != 0:
        print("\n-- psql stopped at the error above. Nothing after it ran.")
    return result


setup = subprocess.run(
    PG_PREFIX + [PSQL, "-X", "--set=ON_ERROR_STOP=on", "--quiet",
                 "--dbname", PRACTICE_DB, "--file", "-"],
    input=section_1, text=True, capture_output=True,
)
print(setup.stdout.strip("
"))
if setup.returncode != 0:
    print(setup.stderr)
    raise RuntimeError("SECTION 1 did not finish. Read the error above.")
print("Practice database:", PRACTICE_DB)

## Step 1: Part 1, Check the Data

### 1a. Join each ticket to its requester

A ticket's `requester_id` matches a `user_id` in `users`. Fill in the two blanks
in the `ON` line, then run the cell.

**You should see** **12 rows**, one per ticket, from 1001 with Requester 101 to
1012 with Requester 104.

Copy your finished query into `midterm.sql` under the SECTION 1 TODO. Keep the
`MY_JOIN_SQL` name: Step 4 reuses it.

In [ ]:
MY_JOIN_SQL = """
SELECT t.ticket_id, u.display_name AS requester
FROM midterm_metro.tickets AS t
JOIN midterm_metro.users AS u ON ______ = ______
ORDER BY t.ticket_id
"""
run_sql(MY_JOIN_SQL + ";")

### 1b. Prove three integrity rules work

The midterm asks you to explain three **non-key** integrity controls. Primary
keys and foreign keys do not count. Scroll up to the `CREATE TABLE` statements
in the starter (or open `midterm.sql`) and find:

- a **NOT NULL** column,
- a **UNIQUE** column, and
- a **CHECK** rule.

On Day 1 you tested a rule by making a write that **should** fail, then checking
that it failed for the right reason. Do the same here. Make every value valid
except the one that breaks the rule, so nothing else can cause the error.

| Rule type | SQLSTATE you should see |
|---|---|
| NOT NULL | 23502 |
| UNIQUE | 23505 |
| CHECK | 23514 |

Each test runs inside `BEGIN` and `ROLLBACK`, so nothing is saved. Here is a
worked example for the UNIQUE email rule. User 999 does not exist, but the email
already belongs to user 101.

**You should see** `ERROR:  23505: duplicate key value violates unique
constraint "users_email_key"`.

In [ ]:
run_sql("""
BEGIN;
INSERT INTO midterm_metro.users
    (user_id, display_name, email, role, neighborhood, created_at)
VALUES
    (999, 'Test User', 'requester101@example.test', 'resident', 'Harbor', now());
ROLLBACK;
""")

**Your turn: a NOT NULL test.** Pick a NOT NULL column that is not a key. Write
an INSERT that leaves only that column NULL. Use ticket ID **1099** or user ID
**999**, which do not exist. If you insert a ticket, use requester 101, which does
exist. **You should see** SQLSTATE **23502** naming your column.

In [ ]:
run_sql("""
BEGIN;
-- TODO: write an INSERT that breaks only one NOT NULL rule.
ROLLBACK;
""")

**Your turn: a CHECK test.** The `tickets` table has a CHECK rule. Read it, then
write an INSERT that breaks only that rule. **You should see** SQLSTATE **23514**
and the constraint name.

In [ ]:
run_sql("""
BEGIN;
-- TODO: write an INSERT that breaks only the CHECK rule on tickets.
ROLLBACK;
""")

**Write in README, section "Database checks":** your counts, what your join
returned, and the three controls. For each control, give its column and the bad
data it prevents. You may add your three tests to `midterm.sql` as evidence, but
**comment them out** so the file still runs top to bottom.

## Step 2: Part 2, Roll Back, Then Commit

Ticket **1004** starts with priority **low**.

### 2a. The supplied rollback example

This is SECTION 2A from the starter, unchanged. Run it once.

**You should see** `low`, then `high` inside the transaction, then `low` again
after ROLLBACK.

In [ ]:
run_sql("""
SELECT ticket_id, priority FROM midterm_metro.tickets WHERE ticket_id = 1004;
BEGIN;
UPDATE midterm_metro.tickets SET priority = 'high' WHERE ticket_id = 1004;
SELECT ticket_id, priority FROM midterm_metro.tickets WHERE ticket_id = 1004;
ROLLBACK;
SELECT ticket_id, priority FROM midterm_metro.tickets WHERE ticket_id = 1004;
""")

### 2b. Your commit example

Write the same six statements, but change `'high'` to `'medium'` and `ROLLBACK`
to `COMMIT`. Run the cell **once**.

**You should see** `low`, then `medium`, then `medium` after COMMIT.

If you run it twice, the first value will already be `medium`. To start over,
rerun the Step 0 starter cell, then 2a, then this cell.

Copy your six statements into SECTION 2B of `midterm.sql`. Leave the committed
`medium` in place.

In [ ]:
run_sql("""
-- TODO: SELECT the starting priority of ticket 1004.
-- TODO: BEGIN, UPDATE the priority to 'medium', SELECT again,
--       COMMIT, and SELECT one more time.
""")

**Write in README, section "Transactions":** the six values you observed, in the
template's table, and why the final values differ.

## Step 3: Part 4, Test One Improvement

This is the recommended index route from the starter. If you chose a different
improvement in Week 7, keep your choice and use this time to run its before and
after checks instead.

### 3a. Before the index (SECTION 4A)

**You should see** the ordered IDs **1011, 1007, 1001** and a plan line that
names the scan, most likely `Seq Scan on tickets`.

In [ ]:
run_sql("""
SELECT ticket_id, subject, opened_at
FROM midterm_metro.tickets
WHERE status = 'open'
ORDER BY opened_at DESC, ticket_id DESC
LIMIT 20;

EXPLAIN (ANALYZE, BUFFERS)
SELECT ticket_id, subject, opened_at
FROM midterm_metro.tickets
WHERE status = 'open'
ORDER BY opened_at DESC, ticket_id DESC
LIMIT 20;
""")

### 3b. Add the index (SECTION 4B)

Run this **once**. If PostgreSQL says the index already exists, rerun the Step 0
starter cell and repeat Steps 2 and 3.

In [ ]:
run_sql("""
CREATE INDEX midterm_tickets_status_opened_idx
ON midterm_metro.tickets (status, opened_at DESC, ticket_id DESC);
""")

### 3c. After the index (SECTION 4C)

The query is identical. **You should see** the same IDs, **1011, 1007, 1001**,
and an `index_bytes` value, likely **16384** (16 kB).

With only 12 tickets, PostgreSQL may keep the sequential scan. **That is a
correct result.** Do not add fake rows to force a different plan.

In [ ]:
run_sql("""
SELECT ticket_id, subject, opened_at
FROM midterm_metro.tickets
WHERE status = 'open'
ORDER BY opened_at DESC, ticket_id DESC
LIMIT 20;

EXPLAIN (ANALYZE, BUFFERS)
SELECT ticket_id, subject, opened_at
FROM midterm_metro.tickets
WHERE status = 'open'
ORDER BY opened_at DESC, ticket_id DESC
LIMIT 20;

SELECT pg_relation_size(
  'midterm_metro.midterm_tickets_status_opened_idx'
) AS index_bytes;
""")

**Write in README, section "One improvement":** the ordered IDs before and
after, the scan line from each plan, the index size, whether the plan changed,
whether the result stayed the same, and one cost of keeping the index.

## Step 4: Part 5, Back Up, Restore Somewhere Separate, and Check

This is the Day 1 recovery drill on **your midterm data**. It backs up the
`midterm_metro` schema with `pg_dump`, restores it into a **separate, empty**
database with `pg_restore`, and runs the same checks on both databases.

### 4a. Write your checks first

Before you run anything, write at least three checks and the result you expect
from each. A check you plan before you look is a real test. The midterm page
suggests these, but use your own values:

| Check | Expected result |
|---|---|
| Table counts | 8 users, 12 tickets, 21 ticket events |
| Requester relationships | your join returns 12 rows |
| Ticket 1004 | priority medium, after your Part 2 commit |
| The added index | midterm_tickets_status_opened_idx exists |

Double-click the next cell and replace its TODO with your own table.

**My recovery checks and expected results**

TODO: write at least three checks, each with its expected result.

### 4b. Back up and restore

`--format=custom` makes an archive for `pg_restore`. `--schema=midterm_metro`
limits the backup to the midterm schema. The restore target is a new, empty
database with a different name. `--exit-on-error` and `--single-transaction`
stop a failed restore from looking successful or leaving half the objects
behind.

**You should see** a list of the archive's tables, then the name of the
separate restore database.

In [ ]:
if DUMP_FILE.exists():
    DUMP_FILE.unlink()
subprocess.run(
    PG_PREFIX + [PG_DUMP, "--format=custom", "--schema=midterm_metro",
                 "--no-privileges", "--file", str(DUMP_FILE), PRACTICE_DB],
    check=True,
)
archive_list = subprocess.run(
    PG_PREFIX + [PG_RESTORE, "--list", str(DUMP_FILE)],
    check=True, text=True, capture_output=True,
)
print("\n".join(line for line in archive_list.stdout.splitlines()
                if any(term in line for term in ("SCHEMA", "TABLE ", "TABLE DATA", "INDEX"))))

subprocess.run(PG_PREFIX + [DROPDB, "--if-exists", RESTORE_DB], check=True, capture_output=True)
subprocess.run(PG_PREFIX + [CREATEDB, RESTORE_DB], check=True)
subprocess.run(
    PG_PREFIX + [PG_RESTORE, "--exit-on-error", "--single-transaction",
                 "--no-owner", "--no-privileges", "--dbname", RESTORE_DB, str(DUMP_FILE)],
    check=True,
)
print("\nRestored into the separate database:", RESTORE_DB)

### 4c. Run the same checks on both databases

The cell runs your checks against the practice database and the restored copy,
then prints them side by side. Each line should say **MATCH**. Then compare
each restored value with the expected result **you** wrote in 4a.

If you chose a different improvement in Part 4, replace the index check with a
check for your change.

In [ ]:
recovery_checks = [
    ("users", "SELECT count(*) FROM midterm_metro.users"),
    ("tickets", "SELECT count(*) FROM midterm_metro.tickets"),
    ("ticket_events", "SELECT count(*) FROM midterm_metro.ticket_events"),
    ("requester join rows", f"SELECT count(*) FROM ({MY_JOIN_SQL}) AS joined"),
    ("ticket 1004 priority",
     "SELECT priority FROM midterm_metro.tickets WHERE ticket_id = 1004"),
    ("index exists (1=yes)",
     "SELECT count(*) FROM pg_indexes WHERE schemaname = 'midterm_metro' "
     "AND indexname = 'midterm_tickets_status_opened_idx'"),
]


def check_value(database, sql_text):
    result = subprocess.run(
        PG_PREFIX + [PSQL, "-X", "--tuples-only", "--no-align",
                     "--dbname", database, "--command", sql_text],
        text=True, capture_output=True,
    )
    return result.stdout.strip() if result.returncode == 0 else "ERROR"


print(f"{'check':<22}{'practice':>10}{'restored':>10}")
for label, sql_text in recovery_checks:
    before = check_value(PRACTICE_DB, sql_text)
    after = check_value(RESTORE_DB, sql_text)
    verdict = "MATCH" if before == after and before != "ERROR" else "DIFFERENT"
    print(f"{label:<22}{before:>10}{after:>10}   {verdict}")

### 4d. What this restore does and does not show

You restored the **Colab practice copy** of `midterm_metro`. If your midterm
database is this notebook, you may report the restore and its checks as
**observed** in your README. If your midterm database is in Supabase, this was
a rehearsal: your README recovery plan for Supabase stays a **proposed plan**
unless you restore that database too. Never write that a restore passed if you
did not run it.

This backup left out roles and grants (`--no-privileges`, `--no-owner`), and the
restore was on the same temporary server as the source. Name those limits in
your README.

**Write in README, section "Recovery plan":** status (proposed or executed),
source, the backup command, a separate destination, the restore command, your
three or more checks with expected results, and the limits. The midterm page has
the Supabase versions of both commands.

## Step 5: Where Are You?

Replace the TODO with two or three sentences: which midterm parts now have
results in your README, which part is still unfinished, and your next step.

TODO

## Finish: Remove This Run's Practice Resources

Keep your output in the notebook, then run this cell. It removes the two
practice databases and the backup file and, in Colab, stops PostgreSQL.

**Submit** the completed notebook in Brightspace: in Colab, **File > Download >
Download .ipynb**.

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

In [ ]:
for database_name in (RESTORE_DB, PRACTICE_DB):
    assert database_name.startswith(("cst4714_midterm_practice_", "cst4714_midterm_restore_"))
    subprocess.run(PG_PREFIX + [DROPDB, "--if-exists", database_name],
                   check=True, text=True, capture_output=True)
shutil.rmtree(ARTIFACT_DIR)
if IN_COLAB:
    subprocess.run(["service", "postgresql", "stop"], check=True)
del PRACTICE_DB, RESTORE_DB
print("Removed this run's practice databases and backup file.")